<a href="https://colab.research.google.com/github/CardiacModelling/summer-school-2026/blob/main/hands-on-session-0-installation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Myokit installation and example simulation

In this notebook, we
- install [Myokit](https://myokit.org) in Google Colab
- run an example simulation
- explain the example, with links to the [docs](https://myokit.readthedocs.io/en/stable/)

To install Myokit, we
1. Use `apt-get` to install the required [Sundials](https://computing.llnl.gov/projects/sundials) library
2. Use `pip` to install the package `myokit`

In [ ]:
!apt-get install libsundials-dev
!pip install myokit

Next, we import `myokit` module and make our first call:

In [ ]:
import myokit
print(myokit.version())

That works! For the real test, we'll load the built-in example model and run a 1000ms simulation:

In [ ]:
model, protocol, script = myokit.load('example')
simulation = myokit.Simulation(model, protocol)
log = simulation.run(1000)

To visualise the results, we'll use [matplotlib](https://matplotlib.org/):

In [ ]:
import matplotlib.pyplot as plt
plt.plot(log.time(), log['membrane.V'])
plt.show()

## Tweak: getting rid of the debug info

When we created the simulation, it generated a lot of debug output:
```
INFO:root:running build_ext
INFO:root:building 'myokit_sim_1_5499951616817020280' extension
INFO:root:creating build/temp.linux-x86_64-cpython-312/tmp/tmpxss9t8mumyokit
INFO:root:x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/include -I/usr/local/lib/python3.12/dist-packages/myokit/_sim -I/usr/local/lib/python3.12/dist-packages/myokit/_sim -I/usr/include/python3.12 -c /tmp/tmpxss9t8mumyokit/source.c -o build/temp.linux-x86_64-cpython-312/tmp/tmpxss9t8mumyokit/source.o
INFO:root:creating build/lib.linux-x86_64-cpython-312
INFO:root:x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/tmp/tmpxss9t8mumyokit/source.o -L/usr/local/lib -L/usr/local/lib64 -L/usr/lib/x86_64-linux-gnu -Wl,--enable-new-dtags,-rpath,/usr/local/lib -Wl,--enable-new-dtags,-rpath,/usr/local/lib64 -lsundials_cvodes -lsundials_nvecserial -lm -o build/lib.linux-x86_64-cpython-312/myokit_sim_1_5499951616817020280.cpython-312-x86_64-linux-gnu.so
INFO:root:copying build/lib.linux-x86_64-cpython-312/myokit_sim_1_5499951616817020280.cpython-312-x86_64-linux-gnu.so ->
```
This doesn't help us much, and can look a bit daunting.
We can get rid of these lines with:

In [ ]:
import logging
logging.disable(logging.INFO)

And run again:

In [ ]:
model, protocol, script = myokit.load('example')
simulation = myokit.Simulation(model, protocol)
log = simulation.run(1000)

fig = plt.figure()
ax = fig.add_subplot()
ax.plot(log.time(), log['membrane.V'])
plt.show()

## The same example, line by line

If you'd like a bit more insight into what we just did, we'll now run through the same example code more slowly, explaining each line.

In [ ]:
model, protocol, script = myokit.load('example')

First, we loaded a [Model](https://myokit.readthedocs.io/en/stable/api_core/Model.html), a [Pacing protocol](https://myokit.readthedocs.io/en/stable/api_simulations/Protocol.html), and a script.

Usually, we'd load these from an `.mmt` file, for example
```
model, protocol, script = myokit.load('my-model.mmt')
```
but here we used the built-in `example` file instead.

The **model** contains all the equations that make up the cell or channel model, along with its initial conditions.

A selection of example models is hosted [here](https://github.com/myokit/models).
To get a feel for what's inside a model, try the [1977 model by Beeler & Reuter](https://github.com/myokit/models/blob/main/c/beeler-1977.mmt).
Then, if you're feeling brave, check out a [modern model](https://github.com/myokit/models/blob/main/c/tomek-2020.mmt).
The model syntax is explained briefly below.

A **protocol** is used to specify an external input to a model.
For action potential models, this is usually a periodic stimulus current.
For ion channel models, this is typically a voltage-clamping protocol.

We can inspect the protocol we loaded above with

In [ ]:
print(protocol.code())

This specifies a stimulus of size `1`, starting at 50 time units - let's say milliseconds, lasting 0.5ms, and repeating indefinitely every 1000ms.
To cap the number of repeats, we can set the "multiplier" to a positive number.

The **script** part of an mmt file can be used to bundle an example simulation with a model.
We won't be using this here.

Next, we create a [Simulation](https://myokit.readthedocs.io/en/stable/api_simulations/Simulation.html) that uses the loaded model and protocol:

In [ ]:
simulation = myokit.Simulation(model, protocol)

This step compiles the `mmt` code, which can take a bit longer than the other lines, but should complete within a second or so.
When performance matters, for example when fitting a model or running sensitivity analyses, it's best to avoid creating new simulations and re-use existing ones instead.

Finally, we [run](https://myokit.readthedocs.io/en/stable/api_simulations/Simulation.html#myokit.Simulation.run) the simulation for 1000ms and receive the results in a [Data log](https://myokit.readthedocs.io/en/stable/api_simulations/DataLog.html).
This is a Python [dict](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) with some added features.

In [ ]:
log = simulation.run(1000)

We can see what's inside by printing its `keys`:

In [ ]:
print(log.keys())

Each key points to an array of logged points:

In [ ]:
print(log['membrane.V'])

The keys corresponds to a model variable, and typically consist of two parts: a component (e.g. `membrane`) and a variable name (e.g. `V`).

In this model, the variable representing time is called `engine.time`, but we can also access it using the special `time()` method:

In [ ]:
print(log.keys())

In [ ]:
print(log['engine.time'])
print(log.time())

Finally, we use the obtained datalog to plot the results.

In this example, we use Matplotlib - for which there are many tutorials online.
In short, we create a figure, add an axes, plot the variable `membrane.V` against time on that axes, and then display the result:

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(log.time(), log['membrane.V'])
plt.show()

## Model syntax

Finally, let's take a look at the model syntax.
To do this, we'll print out the model code:

In [ ]:
print(model.code())

Myokit models are split into _Components_ indicated by square brackets. For example `[membrane]`.

Within a component, we can create variables by writing their definining equation. For example
```
x = 1
```
creates a constant `x` with a value of 1.

We can indicate the units a variable is in using the `in` keyword, written indented on the next line:
```
x = 1
  in [m/s]
```
and we can assign units to numbers using a similar syntax:
```
x = 1 [m] / 1 [s]
  in [m/s]
```

_State variables_ are created by writing their defining equation as an ODE:
```
dot(y) = 10
```
The initial values of all state variables must be specified in the model header, at the top of the file. (See the model code print-out above for examples).

As before, the `in` keyword can be used to set the units of `y` (_not_ the units of `dy/dt`). If time is in `s`, we can write:
```
dot(y) = 10 [m/s]
  in [m]
```

Finally, meta data can be added using a `keyword: value` syntax, for example
```
K_o = 4 [mM]
  in [mM]
  description: The external potassium concentration
```
and model inputs and outputs are specified using the `bind` and `label` keywords - but these won't be discussed here.

A full description of the syntax is given in Myokit's [documentation](https://myokit.readthedocs.io/en/stable/syntax/index.html).